In [86]:
# Library

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression, SelectFromModel
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [67]:
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target

In [68]:
df = df.drop(['Latitude', 'Longitude'], axis=1)

In [69]:
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,3.422


In [70]:
# Cek missing value

df.isna().sum()

MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
MedHouseVal    0
dtype: int64

In [ ]:
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ('selection', SelectKBest(
        score_func=f_regression,
        k=3
    )), # feature selection dulu baru scaling agar tidak menimbulkan data leakage
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred)))

RMSE: 0.7797547297409738


In [ ]:
# add multiple model -> cross validation -> hyperparameter tuning -> train the best model and best parameters

transform_pipeline = Pipeline([
    ('selection', SelectKBest(
        score_func=f_regression, #f_regression menghitung hubungan linear antara setiap fitur dan target menggunakan F-statistic.
        k=4 #Mempertahankan 4 fitur dengan skor tertinggi
    )),
    ('scaler', StandardScaler())
])

models = {
    'LinearRegression': LinearRegression(),
    'RidgeRegression': Ridge(),
    'LassoRegression': Lasso(),
    'DecisionTree': DecisionTreeRegressor(),
    'RandomForest': RandomForestRegressor()

}

for name, model in models.items():
    full_pipeline = Pipeline([
        ('preprocessing', transform_pipeline),
        ('model', model)
    ])

    scores = cross_val_score(full_pipeline, X_train, y_train, cv=5, scoring='neg_mean_squared_error') #mengembalikan negatif dari Mean Squared Error (MSE) karena konvensi scoring-nya menganggap skor yang lebih tinggi selalu lebih baik
    rmse_cv = np.sqrt(-scores).mean() #Tanda minus mengubah MSE negatif menjadi positif dan menghitung rata-rata RMSE dari kelima fold.

    full_pipeline.fit(X_train, y_train)
    y_pred = full_pipeline.predict(X_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"{name}")
    print('CV RMSE:', rmse_cv)
    print('Test RMSE', rmse_test)

LinearRegression
CV RMSE: 0.8028632656242548
Test RMSE 0.7569523819083416
RidgeRegression
CV RMSE: 0.802854498712111
Test RMSE 0.7569602246466123
LassoRegression
CV RMSE: 1.1562699357941972
Test RMSE 1.144774789328518
DecisionTree
CV RMSE: 0.9893410614922462
Test RMSE 0.9761043420108785
RandomForest
CV RMSE: 0.725954824599411
Test RMSE 0.7004269920768683


In [ ]:
pipeline = Pipeline([
    ('selection', SelectKBest(score_func=f_regression, k=4)),
    ('scaler', StandardScaler()),
    ('model', LinearRegression()) # pipeline harus memiliki estimator akhir agar bisa melakukan training dan prediksi Namun, tidak perlu memasukkan semua model sekaligus ke dalam Pipeline. Cukup gunakan satu model sebagai nilai awal. Jadi, model awal LinearRegression() bisa diganti dengan Lasso(), Ridge(), atau model lainnya selama estimator tersebut kompatibel dengan pipeline.m
])

search_space = [
    {
        'model': [LinearRegression()]
    },
    {
        'model': [Lasso()],
        'model__alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100], #Kekuatan regularisasi, Lebih besar → lebih banyak koefisien dapat menjadi nol
        'model__max_iter': [10000, 2000, 50000], #Maksimum iterasi optimasi, Tambah jika model belum konvergen
        'model__tol': [0.0001, 0.001, 0.01] #Toleransi konvergensi, Mengatur kriteria berhenti
    },
    {
        'model': [Ridge()],
        'model__alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000], #Kekuatan regularisasi, Lebih besar → koefisien lebih mengecil
        'model__solver': ['auto', 'svd', 'lsqr', 'lbfgs'] #Algoritma optimasi, Umumnya gunakan default terlebih dahulu. Konvergen artinya proses optimisasi sudah mencapai kondisi yang cukup stabil, sehingga perubahan berikutnya dianggap tidak lagi signifikan
    },
    {
        'model': [DecisionTreeRegressor()],
        'model__max_depth': [None, 3, 5, 10, 15, 20], # Membatasi kedalaman pohon untuk mengurangi overfitting, Lebih besar → pohon lebih kompleks
        'model__min_samples_split': [2, 5, 10, 20], # Minimum sampel untuk membagi node, Lebih besar → pembagian lebih terbatas
        'model__min_samples_leaf': [1, 2, 5, 10], # Minimum sampel di setiap leaf, Lebih besar → prediksi lebih halus
        'model__max_leaf_nodes': [None, 10, 20, 50, 100] # Maksimum jumlah leaf, Lebih kecil → kompleksitas dibatasi
        #'max_features': [] Fitur yang dipertimbangkan saat mencari split, Lebih sedikit fitur → pilihan split lebih terbatas
    },
    {
        'model': [RandomForestRegressor()],
        'model__n_estimators': [100, 200, 300, 500], #Jumlah pohon, Lebih banyak → prediksi biasanya lebih stabil, tetapi komputasi bertambah
        'model__max_depth': [None, 5, 10, 15, 20], #Kedalaman maksimum setiap pohon, Membatasi kompleksitas pohon
        'model__min_samples_split': [2, 5, 10], #Minimum sampel untuk split, Lebih besar → pohon cenderung lebih sederhana
        'model__min_samples_leaf': [1, 2, 4, 8],#Minimum sampel di setiap leaf, Lebih besar → mengurangi variasi prediksi
        'model__max_features': [1.0, 0.7, 0.5] #Jumlah/proporsi fitur yang dipertimbangkan pada setiap split, Mengontrol keragaman antarpohon
    }
]

random_search = RandomizedSearchCV(
    estimator=pipeline, 
    param_distributions=search_space, 
    cv=5, 
    scoring='neg_mean_squared_error', 
    random_state=42, 
    verbose=0
    )

random_search.fit(X_train, y_train)


# Ini makan waktu banyak dan tidak efektif, mending dibagi dulu jangan langsung semua model

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('selection',
                                              SelectKBest(k=4,
                                                          score_func=<function f_regression at 0x12dd17ba0>)),
                                             ('scaler', StandardScaler()),
                                             ('model', LinearRegression())]),
                   param_distributions=[{'model': [LinearRegression()]},
                                        {'model': [Lasso()],
                                         'model__alpha': [0.0001, 0.001, 0.01,
                                                          0.1, 1, 10, 100],
                                         'model__max_iter': [10000, 2000,
                                                             50000],
                                         'model_...
                                                                   50, 100],
                                         'model__min_samples_leaf': [1, 2, 5,
                                                                     10],
                                         'model__min_samples_split': [2, 5, 10,
                                                                      20]},
                                        {'model': [RandomForestRegressor()],
                                         'model__max_depth': [None, 5, 10, 15,
                                                              20],
                                         'model__max_features': [1.0, 0.7, 0.5],
                                         'model__min_samples_leaf': [1, 2, 4,
                                                                     8],
                                         'model__min_samples_split': [2, 5, 10],
                                         'model__n_estimators': [100, 200, 300,
                                                                 500]}],
                   random_state=42, scoring='neg_mean_squared_error')

In [77]:
best_model = random_search.best_estimator_
print('Best Params:', random_search.best_params_)

y_pred = best_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print('Test RMSE:', rmse)

Best Params: {'model__n_estimators': 500, 'model__min_samples_split': 2, 'model__min_samples_leaf': 8, 'model__max_features': 0.7, 'model__max_depth': 15, 'model': RandomForestRegressor()}
Test RMSE: 0.6801898717523585


In [ ]:
# Contoh pipeline yang efektif dan efisien

pipelines = {
    'LinearRegression': Pipeline([
        ('selection', SelectKBest(f_regression, k=4)),
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ]),
    'Ridge': Pipeline([
        ('selection', SelectKBest(f_regression, k=4)),
        ('scaler', StandardScaler()),
        ('model', Ridge())
    ]),
    'Lasso': Pipeline([
        ('selection', SelectKBest(f_regression, k=4)),
        ('scaler', StandardScaler()),
        ('model', Lasso())
    ]),
    'DecisionTree': Pipeline([
        ('selection', SelectKBest(f_regression, k=4)),
        ('scaler', StandardScaler()),
        ('model', DecisionTreeRegressor(random_state=42))
    ]),
    'RandomForest': Pipeline([
        ('selection', SelectKBest(f_regression, k=4)),
        ('scaler', StandardScaler()),
        ('model', RandomForestRegressor(random_state=42))
        #Scaling untuk Decision Tree dan Random Forest sebenarnya tidak diperlukan, meskipun penggunaan scaling di pipeline kamu bukan masalah besar.
    ]),
    'GradientBoosting': Pipeline([
        ('selection', SelectKBest(f_regression, k=4)),
        ('scaler', StandardScaler()),
        ('model', GradientBoostingRegressor(random_state=42))
    ])
}

search_spaces = {
    'LinearRegression': {},
    'Ridge': {
        'model__alpha': [0.01, 0.1, 1, 10, 100]
    },
    'Lasso': {
        'model__alpha': [0.001, 0.01, 0.1, 1, 10]
    },
    'DecisionTree': {
        'model__max_depth': [None, 3, 5, 10, 15],
        'model__min_samples_leaf': [1, 2, 5, 10]
        # Mengontrol kompleksitasnya menggunakan max_depth, min_samples_split, dan min_samples_leaf.
    },
    'RandomForest': {
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [None, 5, 10, 15],
        'model__min_samples_leaf': [1, 2, 4]
        # Menambah n_estimators umumnya membuat prediksi lebih stabil, sedangkan max_depth terlalu besar dapat meningkatkan risiko overfitting.
    },
    'GradientBoosting': {
        'model__n_estimators': [50, 100, 200], #jumlah tree yang dibangun.
        'model__learning_rate': [0.01, 0.05, 0.1], #seberapa besar kontribusi setiap tree terhadap model.
        'model__max_depth': [2, 3, 5], #kompleksitas setiap tree.
        'model__min_samples_leaf': [1, 2, 5] #jumlah minimum sampel pada setiap leaf.
        # learning_rate yang lebih kecil biasanya membutuhkan lebih banyak tree. max_depth yang terlalu besar dapat meningkatkan risiko overfitting.
    }
}

results = {}

for name, pipeline in pipelines.items():
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=search_spaces[name],
        n_iter=10 if search_spaces[name] else 1,
        cv=5,
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    results[name] = {
        'best_model': search.best_estimator_,
        'best_params': search.best_params_,
        'cv_rmse': np.sqrt(-search.best_score_)
    }

    print(name)
    print('Best params:', search.best_params_)
    print('CV RMSE:', np.sqrt(-search.best_score_))

In [ ]:
# SelectFromModel -> digunakan untuk feature selection dengan embedded methods

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selection', SelectFromModel(
        RandomForestRegressor(
            n_estimators=100,
            random_state=42
        ),
        threshold='median'
    )),
    ('model', LinearRegression())
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

selector = pipeline.named_steps['selection']
selector_features = X_train.columns[selector.get_support()]

search_space = [
    {
        'selection__threshold': ['mean', 'median'],
        'selection_estimator__n_estimator':[100, 200, 300],
        'model':[
            LinearRegression(), 
            Ridge(),
            Lasso()
        ]
    }
]

print('RMSE:', rmse)
print(selector_features.tolist())



RMSE 0.781776198369133
['MedInc', 'HouseAge', 'AveOccup']
